In [9]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt

In [11]:
df = pd.read_csv("../data/sample/AAPL_historical_sample.csv")
df["Date"] = pd.to_datetime(df["Date"])
df.set_index("Date", inplace=True)

df["Daily Return"] = df["Close"].pct_change()
df["MA20"] = df["Close"].rolling(window=20).mean()
df["MA50"] = df["Close"].rolling(window=50).mean()
df["Target"] = df["Close"].shift(-1)

df.dropna(inplace=True)

/var/folders/w_/j51psmx961g5qp6tmmyx79sr0000gn/T/ipykernel_57475/3393443895.py:2: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  df["Date"] = pd.to_datetime(df["Date"])


In [22]:
df["Lag_1"] = df["Close"].shift(1)
df["Lag_3"] = df["Close"].shift(3)
df["Lag_5"] = df["Close"].shift(5)

df.dropna(inplace=True)

features = ["Close", "MA20", "MA50", "Daily Return", "Lag_1", "Lag_3", "Lag_5"]
X = df[features]
y = df["Target"]

split_index = int(len(df) * 0.8)
X_train, X_test = X[:split_index], X[split_index:]
y_train, y_test = y[:split_index], y[split_index:]

In [24]:
model = xgb.XGBRegressor(
    objective='reg:squarederror',
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='rmse',
    random_state=42
)

model.fit(
    X_train,
    y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)
y_pred = model.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"XGBoost MSE: {mse:.2f}")
print(f"XGBoost R²: {r2:.4f}")

XGBoost MSE: 168.71
XGBoost R²: 0.4479


## Why XGBoost Underperformed Compared to Linear Regression

- Because stock market data is a time-series data, a model respecting sequential patters would be better
- XGBoost is a tree-based model and it tends to overfit on smaller datasets
- Stock prices and features are highly correlated and have linear relationships 

**This experiment helped highlight the importance of model selection and evaluation.**